# StreamFlix — SQL Analysis

**Analyst:** Meghana J
**Engine:** SQLite (in-memory), queried from Python
**Inputs:** the six StreamFlix CSV files and the provided `schema_and_sql.sql`

**Purpose.** The business questions in this project can be answered two ways — in pandas or in SQL — and a real analyst is expected to do both. This notebook loads the six files into a proper relational schema and answers every business question in SQL, including the twelve analytical queries supplied with the project, plus five of my own that target the findings from Phases 1 to 3.

The last section cross-checks the SQL results against the Python KPI calculations. If the two agree to the decimal, the numbers in the dashboard and the report are verified by two independent methods.

**Structure**

| Section | Content |
|---|---|
| 1 | Build the schema and load 979,000 rows |
| 2 | The twelve supplied queries, each with an observation |
| 3 | Five queries of my own: where churn lives, format efficiency, data integrity |
| 4 | Cross-check against the Python KPIs |
| 5 | What SQL added to the analysis |

## 1. Schema and load

The `CREATE TABLE` statements come straight from the supplied `schema_and_sql.sql`, so the column types, keys and foreign keys match the project's specification rather than whatever pandas guessed. Rows are then inserted from the CSV files.

Two practical notes. SQLite is dynamically typed, so `DECIMAL` and `BOOLEAN` are stored with their declared affinity and behave as expected in arithmetic and in `CASE WHEN`. And `NULL` churn dates survive the load as `NULL` rather than the string `"nan"`, which matters for every churn query below.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import re
import glob

DATA_DIR = '.'
pd.set_option('display.width', 140)

conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys = ON;')

# --- create the tables using the supplied DDL ---
schema_path = sorted(glob.glob(f'{DATA_DIR}/schema_and_sql*.sql'))[0]
print('schema file:', schema_path)
ddl = open(schema_path, encoding='utf-8').read()
create_block = ddl.split('-- 2. TWELVE ANALYTICAL QUERIES')[0]
statements = re.findall(r'CREATE TABLE\s+\w+\s*\([^;]*\)\s*;', create_block, re.IGNORECASE | re.DOTALL)
for stmt in statements:
    conn.execute(stmt)
print('tables created:', len(statements))

# --- load the CSVs ---
files = {
    'subscribers': 'subscribers.csv',
    'titles': 'titles.csv',
    'watch_history': 'watch_history.csv',
    'ratings': 'ratings.csv',
    'reviews': 'reviews.csv',
    'watchlist': 'watchlist.csv',
}

def load(table, path):
    df = pd.read_csv(path)
    # dates to ISO strings, NaN to None so SQLite stores a true NULL
    for c in df.columns:
        if 'date' in c:
            df[c] = pd.to_datetime(df[c], errors='coerce').dt.strftime('%Y-%m-%d')
    df = df.astype(object).where(pd.notna(df), None)
    placeholders = ','.join(['?'] * len(df.columns))
    conn.executemany(
        f'INSERT INTO {table} ({",".join(df.columns)}) VALUES ({placeholders})',
        df.values.tolist())
    return len(df)

for table, path in files.items():
    print(f'{table:<15} {load(table, f"{DATA_DIR}/{path}"):>9,} rows loaded')

conn.commit()

schema file: ./schema_and_sql (2).sql
tables created: 6


subscribers        15,000 rows loaded
titles              9,000 rows loaded


watch_history     650,000 rows loaded


ratings           130,000 rows loaded


reviews           110,000 rows loaded


watchlist          65,000 rows loaded


In [2]:
# sanity check: the row counts SQL sees must match the files
check = pd.read_sql('''
SELECT 'subscribers' AS table_name, COUNT(*) AS rows FROM subscribers
UNION ALL SELECT 'titles', COUNT(*) FROM titles
UNION ALL SELECT 'watch_history', COUNT(*) FROM watch_history
UNION ALL SELECT 'ratings', COUNT(*) FROM ratings
UNION ALL SELECT 'reviews', COUNT(*) FROM reviews
UNION ALL SELECT 'watchlist', COUNT(*) FROM watchlist
''', conn)
check.loc[len(check)] = ['TOTAL', check['rows'].sum()]
print(check.to_string(index=False))

   table_name   rows
  subscribers  15000
       titles   9000
watch_history 650000
      ratings 130000
      reviews 110000
    watchlist  65000
        TOTAL 979000


**Loaded.** 979,000 rows across six tables, exactly matching the source files. Every query below runs against this database, not against a pandas frame.

A helper is defined once so each query prints a clean table and its row count without repeating boilerplate.

In [3]:
def q(label, sql, n=12):
    """Run a query, print a labelled table, return the DataFrame."""
    df = pd.read_sql(sql, conn)
    print(f'--- {label} ({len(df)} rows) ---')
    display(df.head(n))
    return df

## 2. The twelve supplied queries

These are the queries shipped with the project. Each is run as written, followed by the observation the result supports.

### Q1 — Top genres by total watch hours

In [4]:
q1 = q('Q1 genres by watch hours', '''
SELECT t.primary_genre,
       ROUND(SUM(w.watch_duration_min) / 60.0, 1) AS watch_hours,
       COUNT(*) AS plays
FROM watch_history w
JOIN titles t ON t.title_id = w.title_id
GROUP BY t.primary_genre
ORDER BY watch_hours DESC;
''')

--- Q1 genres by watch hours (16 rows) ---


,primary_genre,watch_hours,plays
0,Drama,501884.5,95624
1,Comedy,423149.1,84455
2,Action,393171.9,77300
3,Thriller,307024.7,58589
4,Documentary,291216.9,57786
5,Crime,237435.8,44153
6,Romance,214332.3,44210
7,Animation,180093.9,29208
8,Sci-Fi,154332.2,33157
9,Horror,138033.6,28122


Drama leads with 501,885 hours, then Comedy and Action — the top three take 39.6% of all viewing. The tail is thin: Musical and Stand-Up sit near 1% each. A balanced catalogue rather than a hit-driven one.

### Q2 — Origin countries by watch hours and hours per title

In [5]:
q2 = q('Q2 title origin country', '''
SELECT t.country,
       ROUND(SUM(w.watch_duration_min) / 60.0, 1) AS watch_hours,
       COUNT(DISTINCT t.title_id) AS titles,
       ROUND(SUM(w.watch_duration_min) / 60.0 / COUNT(DISTINCT t.title_id), 1) AS hours_per_title
FROM watch_history w
JOIN titles t ON t.title_id = w.title_id
GROUP BY t.country
ORDER BY watch_hours DESC;
''', n=10)

--- Q2 title origin country (20 rows) ---


,country,watch_hours,titles,hours_per_title
0,United States,822684.7,2265,363.2
1,India,464337.8,1163,399.3
2,United Kingdom,282946.4,769,367.9
3,South Korea,223476.4,636,351.4
4,Japan,207130.9,556,372.5
5,Brazil,176353.6,447,394.5
6,Mexico,157176.9,441,356.4
7,Spain,143656.4,419,342.9
8,France,115011.8,377,305.1
9,Canada,107073.0,297,360.5


This is the **production** country of the content, not the subscriber's country — a distinction worth stating out loud, because the two are easy to confuse in a report. The US leads on volume; the hours-per-title column is the more interesting one, since it separates markets that are big from markets that are efficient.

### Q3 — Engagement by release year

In [6]:
q3 = q('Q3 release year', '''
SELECT t.release_year,
       COUNT(DISTINCT t.title_id) AS titles,
       ROUND(SUM(w.watch_duration_min) / 60.0, 1) AS watch_hours,
       ROUND(SUM(w.watch_duration_min) / 60.0 / COUNT(DISTINCT t.title_id), 1) AS hours_per_title
FROM watch_history w
JOIN titles t ON t.title_id = w.title_id
GROUP BY t.release_year
ORDER BY t.release_year;
''', n=20)

--- Q3 release year (58 rows) ---


,release_year,titles,watch_hours,hours_per_title
0,1969,66,24762.5,375.2
1,1970,58,16432.8,283.3
2,1971,55,20331.8,369.7
3,1972,59,18942.7,321.1
4,1973,55,17519.0,318.5
5,1974,43,6978.1,162.3
6,1975,68,23483.6,345.3
7,1976,59,18484.2,313.3
8,1977,63,16426.2,260.7
9,1978,54,9206.0,170.5


Content age turns out to be a weak driver. Hours per title drift only mildly — the 2020s catalogue averages 418 hours per title against 278 for the 1970s — and the pattern is not even monotonic, since the 1960s catalogue averages 375. A 1969 title earns about as much as a 2025 one, which is not how a real catalogue behaves; treat this as a property of the dataset rather than a reason to buy old films.

### Q4 — Movies versus TV shows

In [7]:
q4 = q('Q4 movies vs series', '''
SELECT t.type,
       ROUND(SUM(w.watch_duration_min) / 60.0, 1) AS watch_hours,
       COUNT(*) AS plays,
       ROUND(AVG(w.completion_pct), 1) AS avg_completion_pct
FROM watch_history w
JOIN titles t ON t.title_id = w.title_id
GROUP BY t.type
ORDER BY watch_hours DESC;
''')

--- Q4 movies vs series (2 rows) ---


,type,watch_hours,plays,avg_completion_pct
0,TV Show,2882608.9,267670,65.2
1,Movie,450858.8,382330,65.4


Series take 2,882,609 hours against 450,859 for films — 86.5% of all viewing — while completion is almost identical (65.2% against 65.4%). So the format difference is about *how long people stay*, not about whether they finish.

### Q5 — Watch hours by language

In [8]:
q5 = q('Q5 language', '''
SELECT t.language,
       ROUND(SUM(w.watch_duration_min) / 60.0, 1) AS watch_hours,
       COUNT(*) AS plays
FROM watch_history w
JOIN titles t ON t.title_id = w.title_id
GROUP BY t.language
ORDER BY watch_hours DESC;
''')

--- Q5 language (14 rows) ---


,language,watch_hours,plays
0,English,1352815.8,264430
1,Hindi,464337.8,89686
2,Spanish,347738.2,69080
3,Korean,223476.4,44162
4,Japanese,207130.9,38554
5,Portuguese,176353.6,33063
6,French,115011.8,28375
7,German,96934.4,20586
8,Turkish,82720.0,14185
9,Indonesian,71049.9,12003


English dominates, as expected for a US-led catalogue, with Korean and Spanish the largest non-English markets. This is the query that should drive subtitle and dubbing investment: the second and third languages already carry meaningful volume.

### Q6 — Completion rate by genre

In [9]:
q6 = q('Q6 completion by genre', '''
SELECT t.primary_genre,
       ROUND(AVG(w.completion_pct), 1) AS avg_completion_pct,
       SUM(CASE WHEN w.completed THEN 1 ELSE 0 END) AS completed_sessions,
       COUNT(*) AS total_sessions
FROM watch_history w
JOIN titles t ON t.title_id = w.title_id
GROUP BY t.primary_genre
ORDER BY avg_completion_pct DESC;
''')

--- Q6 completion by genre (16 rows) ---


,primary_genre,avg_completion_pct,completed_sessions,total_sessions
0,Musical,66.8,1213,7425
1,Animation,66.4,4812,29208
2,Stand-Up,66.3,1332,7993
3,Fantasy,66.0,4312,26407
4,Sci-Fi,65.8,5290,33157
5,Mystery,65.6,2754,18074
6,Crime,65.5,6791,44153
7,Thriller,65.3,8867,58589
8,Kids & Family,65.3,2821,18338
9,Horror,65.3,4243,28122


Every genre lands between 64.7% and 66.8% — a spread of 2.1 points. Genre does not predict completion, so there is no genre to cut on those grounds. This is the same finding as the Phase 2 chart, now confirmed in SQL.

### Q7 — Engagement by plan

In [10]:
q7 = q('Q7 hours per subscriber by plan', '''
SELECT s.plan_type,
       COUNT(DISTINCT s.subscriber_id) AS subscribers,
       ROUND(SUM(w.watch_duration_min) / 60.0 / COUNT(DISTINCT s.subscriber_id), 1) AS avg_hours_per_subscriber
FROM watch_history w
JOIN subscribers s ON s.subscriber_id = w.subscriber_id
GROUP BY s.plan_type
ORDER BY avg_hours_per_subscriber DESC;
''')

--- Q7 hours per subscriber by plan (3 rows) ---


,plan_type,subscribers,avg_hours_per_subscriber
0,Premium,4442,228.8
1,Basic with Ads,4458,226.9
2,Standard,6098,214.2


Premium and the ad tier watch almost identical amounts per subscriber — 228.8 hours against 226.9 — while Standard is the lowest at 214.2. The gap is under 7% across all three plans, so engagement is not what separates the tiers; price and perceived value are. The ad tier consumes as much streaming as the most expensive plan while paying a third of the price, which is the economics behind the churn finding.

### Q8 — Device usage share

In [11]:
q8 = q('Q8 device share', '''
SELECT w.device,
       COUNT(*) AS sessions,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM watch_history), 1) AS pct_of_sessions
FROM watch_history w
GROUP BY w.device
ORDER BY sessions DESC;
''')

--- Q8 device share (6 rows) ---


,device,sessions,pct_of_sessions
0,Smart TV,221446,34.1
1,Mobile,175605,27.0
2,Laptop,96408,14.8
3,Tablet,65276,10.0
4,Game Console,52322,8.0
5,Streaming Stick,38943,6.0


Smart TV at 34.1% and Mobile at 27.0% — the living room leads but viewing is genuinely mobile. Note the correlated subquery: it is the simplest way to express a share of total, and SQLite runs it once.

### Q9 — Active rate and churn rate

In [12]:
q9 = q('Q9 active and churn rate', '''
SELECT COUNT(*) AS total_subscribers,
       SUM(CASE WHEN is_active THEN 1 ELSE 0 END) AS active,
       ROUND(100.0 * SUM(CASE WHEN is_active THEN 1 ELSE 0 END) / COUNT(*), 1) AS active_rate_pct,
       ROUND(100.0 * SUM(CASE WHEN NOT is_active THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_rate_pct
FROM subscribers;
''')

--- Q9 active and churn rate (1 rows) ---


,total_subscribers,active,active_rate_pct,churn_rate_pct
0,15000,11199,74.7,25.3


74.7% active, 25.3% churn — both targets met. `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` is the portable way to count a condition across every SQL dialect, and worth using in preference to `COUNT(*) FILTER (WHERE ...)` if the code may run on an older engine.

### Q10 — MRR and ARPU

In [13]:
q10 = q('Q10 MRR and ARPU', '''
SELECT ROUND(SUM(monthly_price_usd), 2) AS mrr_usd,
       COUNT(*) AS active_subscribers,
       ROUND(SUM(monthly_price_usd) / COUNT(*), 2) AS arpu_usd
FROM subscribers
WHERE is_active = TRUE;
''')

--- Q10 MRR and ARPU (1 rows) ---


,mrr_usd,active_subscribers,arpu_usd
0,175868.51,11199,15.7


MRR of $175,868.51 from 11,199 active subscribers, ARPU $15.70. The `WHERE is_active = TRUE` clause is the whole point of the query — including churned subscribers would inflate the count and understate ARPU.

### Q11 — Watchlist conversion

In [14]:
q11 = q('Q11 watchlist conversion', '''
SELECT COUNT(*) AS watchlist_entries,
       SUM(CASE WHEN watched THEN 1 ELSE 0 END) AS converted,
       ROUND(100.0 * SUM(CASE WHEN watched THEN 1 ELSE 0 END) / COUNT(*), 1) AS conversion_rate_pct
FROM watchlist;
''')

--- Q11 watchlist conversion (1 rows) ---


,watchlist_entries,converted,conversion_rate_pct
0,65000,30048,46.2


46.2% of saved titles get watched, comfortably above the 40% target. Note this counts *entries*, not unique subscribers — 61 subscriber-title pairs appear more than once, so a per-user conversion rate would be very slightly different.

### Q12 — Content investment efficiency

In [15]:
q12 = q('Q12 hours per $1,000 of licence cost', '''
SELECT t.primary_genre,
       ROUND(SUM(t.license_cost_usd) / 1000.0, 0) AS spend_thousands_usd,
       ROUND(SUM(t.total_watch_hours), 0) AS watch_hours,
       ROUND(SUM(t.total_watch_hours) / (SUM(t.license_cost_usd) / 1000.0), 3) AS hours_per_1k_usd
FROM titles t
GROUP BY t.primary_genre
ORDER BY hours_per_1k_usd DESC;
''')

--- Q12 hours per $1,000 of licence cost (16 rows) ---


,primary_genre,spend_thousands_usd,watch_hours,hours_per_1k_usd
0,Animation,868312.0,180093.0,0.207
1,Kids & Family,582701.0,103119.0,0.177
2,Documentary,1701154.0,291218.0,0.171
3,Action,2312734.0,393172.0,0.170
4,Crime,1456105.0,237436.0,0.163
5,Drama,3204980.0,501885.0,0.157
6,Fantasy,835817.0,127839.0,0.153
7,Comedy,2762030.0,423148.0,0.153
8,Thriller,2034649.0,307025.0,0.151
9,Romance,1437719.0,214333.0,0.149


Roughly 0.13 to 0.21 hours per $1,000 across genres — a narrow band, which is the point: no genre is a disaster and none is a jackpot. The supplied query aggregates from `titles`, so it answers "what did this genre's catalogue return overall", not "what did each session return". Both are valid; they answer different questions.

## 3. Five queries of my own

The supplied twelve cover the brief's business questions. These five exist because of what Phases 1 to 3 found, and none of them can be answered from a single table.

### Q13 — Where churn actually lives

In [16]:
q13 = q('Q13 churn by plan', '''
SELECT plan_type,
       COUNT(*) AS subscribers,
       SUM(CASE WHEN NOT is_active THEN 1 ELSE 0 END) AS churned,
       ROUND(100.0 * SUM(CASE WHEN NOT is_active THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_rate_pct,
       ROUND(SUM(CASE WHEN is_active THEN monthly_price_usd ELSE 0 END), 2) AS active_mrr
FROM subscribers
GROUP BY plan_type
ORDER BY churn_rate_pct DESC;
''')

--- Q13 churn by plan (3 rows) ---


,plan_type,subscribers,churned,churn_rate_pct,active_mrr
0,Basic with Ads,4458,1529,34.3,20473.71
1,Standard,6100,1469,24.1,71734.19
2,Premium,4442,803,18.1,83660.61


**This is the finding the whole report turns on.** The ad tier churns at 34.3%, Standard at 24.1%, Premium at 18.1% — a 16-point spread — while contributing only $20,474 of the $175,869 MRR. Premium is 32.5% of subscribers and 47.6% of revenue.

### Q14 — Format efficiency: hours bought per $1 million

In [17]:
q14 = q('Q14 efficiency by format', '''
SELECT t.type,
       ROUND(SUM(t.license_cost_usd) / 1000000.0, 1) AS spend_millions,
       ROUND(SUM(t.total_watch_hours), 0) AS watch_hours,
       ROUND(SUM(t.total_watch_hours) / (SUM(t.license_cost_usd) / 1000000.0), 1) AS hours_per_1m
FROM titles t
GROUP BY t.type
ORDER BY hours_per_1m DESC;
''')

--- Q14 efficiency by format (2 rows) ---


,type,spend_millions,watch_hours,hours_per_1m
0,TV Show,8411.2,2882610.0,342.7
1,Movie,12803.1,450863.0,35.2


Series return **342.7** watch hours per $1 million of licence cost; films return **35.2** — a 9.7-fold gap, against a genre spread of only 1.6-fold. The supplied Q12 stops at genre, and genre turns out to be the wrong cut: format is where the money is being decided.

### Q15 — Data integrity: viewing after churn

In [18]:
q15 = q('Q15 sessions dated after the subscriber churned', '''
SELECT COUNT(*) AS sessions_after_churn,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM watch_history), 2) AS pct_of_sessions,
       COUNT(DISTINCT w.subscriber_id) AS affected_subscribers
FROM watch_history w
JOIN subscribers s ON s.subscriber_id = w.subscriber_id
WHERE s.churn_date IS NOT NULL
  AND w.watch_date > s.churn_date;
''')

--- Q15 sessions dated after the subscriber churned (1 rows) ---


,sessions_after_churn,pct_of_sessions,affected_subscribers
0,131199,20.18,3708


131,199 sessions — 20.2% of all viewing — are dated after the subscriber churned, across 3,708 subscribers. This is the query that justifies the caveat in every retention discussion: the churn flag cannot be used as a cutoff on activity.

### Q16 — Top titles, with their economics

In [19]:
q16 = q('Q16 top 10 titles', '''
SELECT t.title_name, t.type, t.primary_genre,
       ROUND(SUM(w.watch_duration_min) / 60.0, 0) AS watch_hours,
       ROUND(AVG(w.completion_pct), 1) AS avg_completion,
       ROUND(SUM(t.license_cost_usd) / 1000000.0, 2) AS licence_cost_millions
FROM watch_history w
JOIN titles t ON t.title_id = w.title_id
GROUP BY t.title_id, t.title_name, t.type, t.primary_genre, t.license_cost_usd
ORDER BY watch_hours DESC
LIMIT 10;
''')

--- Q16 top 10 titles (10 rows) ---


,title_name,type,primary_genre,watch_hours,avg_completion,licence_cost_millions
0,Gilded Throne,TV Show,Animation,35443.0,72.0,14690.07
1,The Shadows of Marseille IV,TV Show,Documentary,22538.0,61.0,33729.21
2,The Cipher of Madrid II,TV Show,Comedy,16660.0,67.1,7312.55
3,The Reckoning of Lagos II,TV Show,Drama,14779.0,61.7,205.58
4,Hamburg Frequency IV,TV Show,Action,13963.0,70.8,5337.95
5,Broken Toronto II,TV Show,Thriller,11903.0,62.7,7371.42
6,Jakarta Odyssey IV,TV Show,Kids & Family,11440.0,67.5,1607.91
7,Marseille Circuit II,TV Show,Action,10967.0,65.6,1096.14
8,The Sanctuary of Toronto III,TV Show,Drama,10845.0,71.9,564.79
9,Distant Cairo III,TV Show,Crime,10267.0,70.1,1982.83


**Every title in the top ten is a TV show.** That is the format finding confirmed at the level of individual titles rather than aggregates — and it is the cleanest evidence for the recommendation to shift catalogue spend towards series. Even here no single title dominates: the top 10% of titles take only 31.0% of all plays, which is why the report recommends a portfolio rule rather than chasing a blockbuster.

### Q17 — Subscriber tenure at the point of churn

In [20]:
q17 = q('Q17 tenure at churn by plan', '''
SELECT plan_type,
       COUNT(*) AS churned,
       ROUND(AVG(julianday(churn_date) - julianday(signup_date)) / 30.44, 1) AS avg_tenure_months_at_churn,
       ROUND(MIN(julianday(churn_date) - julianday(signup_date)) / 30.44, 1) AS min_months,
       ROUND(MAX(julianday(churn_date) - julianday(signup_date)) / 30.44, 1) AS max_months
FROM subscribers
WHERE churn_date IS NOT NULL
GROUP BY plan_type
ORDER BY avg_tenure_months_at_churn;
''')

--- Q17 tenure at churn by plan (3 rows) ---


,plan_type,churned,avg_tenure_months_at_churn,min_months,max_months
0,Standard,1469,23.4,1.0,119.1
1,Basic with Ads,1529,23.9,1.0,118.9
2,Premium,803,26.3,1.0,110.9


Churned subscribers leave after roughly two years, and Premium leavers last about 2.4 months longer than the ad tier (26.3 against 23.9 months). So the problem is not *early* churn on the cheap plan — the ad tier holds people about as long as anyone else. The problem is that a much larger share of it eventually leaves. That changes the remedy: it argues for value reinforcement on the ad tier, not an onboarding intervention.

## 4. Cross-check against the Python KPIs

The same numbers were calculated in pandas in Phase 3. If SQL and Python agree, the figures in the dashboard and report are verified by two independent methods.

SQLite's `ROUND` returns a float, and `julianday`-based tenure differs from pandas' month arithmetic by a day or so, so the comparison allows a small tolerance on derived measures and requires exact agreement on sums and counts.

In [21]:
sql_kpis = pd.read_sql('''
SELECT (SELECT ROUND(SUM(watch_duration_min) / 60.0, 2) FROM watch_history) AS total_watch_hours,
       (SELECT COUNT(*) FROM watch_history) AS total_sessions,
       (SELECT ROUND(100.0 * SUM(CASE WHEN is_active THEN 1 ELSE 0 END) / COUNT(*), 2) FROM subscribers) AS active_rate_pct,
       (SELECT ROUND(100.0 * SUM(CASE WHEN NOT is_active THEN 1 ELSE 0 END) / COUNT(*), 2) FROM subscribers) AS churn_rate_pct,
       (SELECT ROUND(AVG(completion_pct), 2) FROM watch_history) AS avg_completion_pct,
       (SELECT ROUND(SUM(monthly_price_usd), 2) FROM subscribers WHERE is_active = TRUE) AS mrr_usd,
       (SELECT ROUND(SUM(monthly_price_usd) / COUNT(*), 2) FROM subscribers WHERE is_active = TRUE) AS arpu_usd,
       (SELECT ROUND(100.0 * SUM(CASE WHEN watched THEN 1 ELSE 0 END) / COUNT(*), 2) FROM watchlist) AS watchlist_conversion_pct
''', conn).iloc[0]

python_kpis = pd.Series({
    'total_watch_hours': 3333467.73,
    'total_sessions': 650000,
    'active_rate_pct': 74.66,
    'churn_rate_pct': 25.34,
    'avg_completion_pct': 65.31,
    'mrr_usd': 175868.51,
    'arpu_usd': 15.70,
    'watchlist_conversion_pct': 46.23,
})

comparison = pd.DataFrame({'SQL': sql_kpis, 'Python (Phase 3)': python_kpis})
comparison['difference'] = (comparison['SQL'].astype(float) - comparison['Python (Phase 3)'].astype(float)).round(2)
comparison['match'] = comparison['difference'].abs() <= 0.01
display(comparison)

print('All KPIs agree within 0.01:', bool(comparison['match'].all()))

,SQL,Python (Phase 3),difference,match
total_watch_hours,3333467.73,3333467.73,0.0,True
total_sessions,650000.00,650000.00,0.0,True
active_rate_pct,74.66,74.66,0.0,True
churn_rate_pct,25.34,25.34,0.0,True
avg_completion_pct,65.31,65.31,0.0,True
mrr_usd,175868.51,175868.51,0.0,True
arpu_usd,15.70,15.70,0.0,True
watchlist_conversion_pct,46.23,46.23,0.0,True


All KPIs agree within 0.01: True


**Verified.** Every headline KPI agrees between the two methods to within 0.01 — watch hours, sessions, active rate, churn rate, completion, MRR, ARPU and watchlist conversion. That is the check that makes the report defensible: nothing in it depends on one library behaving correctly.

## 5. What SQL added

Four things the pandas work either took longer to express or could not show as cleanly:

1. **A declared schema.** The `CREATE TABLE` statements fix the grain and the keys, and `PRAGMA foreign_keys = ON` turns the referential integrity check from a manual comparison into an enforced constraint.
2. **Churn as a conditional aggregate.** `SUM(CASE WHEN NOT is_active THEN 1 ELSE 0 END)` computes a rate inside a single grouped query, which is how a BI tool does it. The same pattern produced Q13, the most important result in the project.
3. **Integrity as a query.** Q15 turns the Phase 1 finding into a repeatable check — 20.2% of sessions after churn — rather than a one-off observation buried in a notebook.
4. **Independent verification.** Section 4 re-derives eight KPIs by a second method. When SQL and pandas agree to 0.01, a mistake in either becomes unlikely rather than merely possible.

One honest limitation: SQLite has no window-function-based cohort table here, and its date arithmetic (`julianday`) differs from pandas' month logic, which is why the cohort retention analysis in Phase 3 stays in Python. The right tool for a calendar-aware cohort curve is either a proper warehouse dialect or pandas — not a compromise between them.